## Production Notebook

In [24]:
import pandas as pd
import joblib

In [25]:
def clean_and_engineer_features(df):

    df = df.copy()

    drop_cols = [
        "subscription_start_date",
        "genres_watched",
        "devices_used",
        "support_ticket_reasons"
    ]

    df = df.drop(columns=[col for col in drop_cols if col in df.columns])

    # Engagement score
    if all(col in df.columns for col in [
        "content_completion_rate",
        "sessions_count"
    ]):

        watch_col = None

        possible_watch_cols = [
            "total_watch_time_hours",
            "watch_time_hours",
            "watch_time"
        ]

        for col in possible_watch_cols:
            if col in df.columns:
                watch_col = col
                break

        if watch_col is not None:
            df["engagement_score"] = (
                df[watch_col] *
                df["content_completion_rate"] *
                df["sessions_count"]
            )
        else:
            df["engagement_score"] = 0

    # Inactivity flag
    if "days_since_last_watch" in df.columns:
        df["inactivity_flag"] = (
            df["days_since_last_watch"] > 14
        ).astype(int)

    # Payment risk
    if all(col in df.columns for col in [
        "payment_failures_count",
        "account_on_hold"
    ]):
        df["payment_risk"] = (
            df["payment_failures_count"] +
            df["account_on_hold"]
        )

    # Support risk
    if "support_tickets_count" in df.columns:
        df["support_risk"] = (
            df["support_tickets_count"] > 0
        ).astype(int)

    # Abandonment rate
    if all(col in df.columns for col in [
        "abandoned_series_count",
        "unique_titles_watched"
    ]):
        df["abandonment_rate"] = (
            df["abandoned_series_count"] /
            (df["unique_titles_watched"] + 1)
        )

    return df

In [26]:
preprocessor = joblib.load("preprocessor.pkl")
model = joblib.load("kmeans_model.pkl")

In [27]:
df_new = pd.read_csv("https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/prod-streaming_churn_dataset_X.csv")

In [28]:
df_new = clean_and_engineer_features(df_new)

In [29]:
X_processed = preprocessor.transform(df_new)

In [30]:
clusters = model.predict(X_processed)

df_new["cluster"] = clusters

In [31]:
df_new["cluster"].value_counts()

cluster
0    2335
1     165
Name: count, dtype: int64

In [32]:
df_new.groupby("cluster")[[
    "engagement_score",
    "abandonment_rate"
]].mean()

,engagement_score,abandonment_rate
cluster,,
0,99.386728,0.652594
1,4323.446122,0.100367


In [33]:
y_true = pd.read_csv("https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/prod-streaming_churn_dataset_y.csv")

In [34]:
y_true = y_true.iloc[:, 0]

In [35]:
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(
    y_true,
    df_new["cluster"]
)

print("Adjusted Rand Index:", round(ari, 4))

Adjusted Rand Index: -0.0339


In [36]:
from sklearn.metrics import normalized_mutual_info_score

nmi = normalized_mutual_info_score(
    y_true,
    df_new["cluster"]
)

print("NMI Score:", round(nmi, 4))

NMI Score: 0.0053


### Production Evaluation Results

The clustering pipeline successfully processed the unseen dataset and generated customer segments without manual modifications. 

When compared with the provided labels, the clustering results showed low alignment scores. This is expected in unsupervised learning because clustering identifies natural behavioral groupings independently of the target labels. The model primarily separated users based on engagement behavior rather than directly optimizing for the provided labels.